DeNovoSeer

#Xiyu Rao
#2026-04-20

In [ ]:
import sys
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import KNNImputer
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

warnings.filterwarnings("ignore")

outdir = "processed_data"
os.makedirs(outdir, exist_ok=True)

Data Loading & Feature Definition

In [ ]:
# replace with your actual coding_annovar_path
coding_annovar_path = '../data/Gene4Denovo/annovar_coding_Gene4Denovo.csv'
# replace with your actual coding_Clinical_path
coding_Clinical_path = '../data/Gene4Denovo/Clinical_coding_Gene4Denovo.csv'

def preprocess_data(annovar_path, Clinical_path):
    annovar_dat = pd.read_csv(annovar_path, na_values=".")
    Clinical_dat = pd.read_csv(Clinical_path, dtype='str', na_values=".")

    index_generator = lambda df: df.apply(
        lambda x: f'chr{x["Chr"]}_start{x["Start"]}_end{x["End"]}_ref{x["Ref"]}_alt{x["Alt"]}',
        axis=1
    )

    annovar_dat['index'] = index_generator(annovar_dat)
    Clinical_dat['index'] = Clinical_dat.apply(
        lambda x: f'chr{x["Chr"]}_start{x["Start"]}_end{x["End"]}_ref{x["Ref"]}_alt{x["Alt"]}',
        axis=1
    )

    if not annovar_dat['index'].equals(Clinical_dat['index']):
        print("ERROR: Index mismatch detected")
        sys.exit(1)

    return pd.merge(annovar_dat, Clinical_dat, on='index', suffixes=('_anno', '_var'))

coding_data = preprocess_data(coding_annovar_path, coding_Clinical_path)

def get_continuous_feature():
    return [
        # conservation
        "Polyphen2_HDIV_score", "Polyphen2_HVAR_score", "LINSIGHT", "GERP++_NR", "GERP++_RS", "phyloP100way_vertebrate",
        "phyloP470way_mammalian", "phyloP17way_primate", "phastCons100way_vertebrate", "phastCons470way_mammalian",
        "phastCons17way_primate", "SiPhy_29way_logOdds",


        # coding features
        "ReVe", "SIFT_score", "SIFT4G_score", "LRT_score", "LRT_Omega", "MutationTaster_score","MutationAssessor_score",
        "FATHMM_score", "PROVEAN_score", "VEST4_score", "MetaSVM_score", "MetaLR_score", "MetaRNN_score", "M-CAP_score",
        "REVEL_score", "MutPred_score", "MutPred2_score", "MVP_score", "gMVP_score", "MPC_score", "PrimateAI_score",
        "DEOGEN2_score", "BayesDel_addAF_score", "BayesDel_noAF_score", "ClinPred_score", "LIST-S2_score",
        "VARITY_R_score", "VARITY_ER_score", "VARITY_R_LOO_score", "VARITY_ER_LOO_score", "ESM1b_score", "EVE_score",
        "AlphaMissense_score", "CADD_raw", "CADD_phred", "DANN_score", "fathmm-MKL_coding_score",
        "fathmm-XF_coding_score", "Eigen-raw_coding", "Eigen-phred_coding", "Eigen-PC-raw_coding",
        "Eigen-PC-phred_coding", "GenoCanyon_score", "integrated_fitCons_score", "integrated_confidence_value",
        "GM12878_fitCons_score", "GM12878_confidence_value", "H1-hESC_fitCons_score", "H1-hESC_confidence_value",
        "HUVEC_fitCons_score", "HUVEC_confidence_value", "bStatistic",


        # Cross-region features
        "Caddnoncoding", "Dannnoncoding", "Fathmm_Mklnoncoding", "Funseq2", "Eigennoncoding", "Eigen_Pc",
        "Genocanyonnoncoding", "Fire", "Remm", "Linsight_Noncoding", "Fitconsnoncoding", "Fathmm_Xf", "Cscape", "Cdts",
        "Dvar", "Fitcons2", "Ncer", "Orion", "Pafa", "Regbase_Reg", "Regbase_Can", "Regbase_Pat", "Divan_Tss",
        "Divan_Region", "VARA_Score",


        # splicing effects
        "SpliceAI", "DS_AG", "DP_AG", "DS_AL", "DP_AL",	"DS_DG", "DP_DG", "DS_DL", "DP_DL",


        # Clinical score
        "Clinical_evidence_score",


        # population-based
        "gnomad41_exome_AF", "gnomad41_exome_AF_raw", "gnomad41_exome_AF_XX", "gnomad41_exome_AF_XY",
        "gnomad41_exome_AF_grpmax", "gnomad41_exome_faf95", "gnomad41_exome_faf99", "gnomad41_exome_fafmax_faf95_max",
        "gnomad41_exome_fafmax_faf99_max", "gnomad41_exome_AF_afr", "gnomad41_exome_AF_amr", "gnomad41_exome_AF_asj",
        "gnomad41_exome_AF_eas", "gnomad41_exome_AF_fin", "gnomad41_exome_AF_mid", "gnomad41_exome_AF_nfe",
        "gnomad41_exome_AF_remaining", "gnomad41_exome_AF_sas", "gnomad41_genome_AF", "gnomad41_genome_AF_raw",
        "gnomad41_genome_AF_XX", "gnomad41_genome_AF_XY", "gnomad41_genome_AF_grpmax", "gnomad41_genome_faf95",
        "gnomad41_genome_faf99", "gnomad41_genome_fafmax_faf95_max", "gnomad41_genome_fafmax_faf99_max",
        "gnomad41_genome_AF_afr", "gnomad41_genome_AF_ami", "gnomad41_genome_AF_amr", "gnomad41_genome_AF_asj",
        "gnomad41_genome_AF_eas", "gnomad41_genome_AF_fin", "gnomad41_genome_AF_mid", "gnomad41_genome_AF_nfe",
        "gnomad41_genome_AF_remaining", "gnomad41_genome_AF_sas", "ExAC_ALL", "ExAC_AFR", "ExAC_AMR", "ExAC_EAS",
        "ExAC_FIN", "ExAC_NFE", "ExAC_OTH", "ExAC_SAS", "ExAC_nontcga_ALL", "ExAC_nontcga_AFR", "ExAC_nontcga_AMR",
        "ExAC_nontcga_EAS", "ExAC_nontcga_FIN", "ExAC_nontcga_NFE", "ExAC_nontcga_OTH", "ExAC_nontcga_SAS",
        "ExAC_nonpsych_ALL", "ExAC_nonpsych_AFR", "ExAC_nonpsych_AMR", "ExAC_nonpsych_EAS", "ExAC_nonpsych_FIN",
        "ExAC_nonpsych_NFE", "ExAC_nonpsych_OTH", "ExAC_nonpsych_SAS", "Kaviar_AF", "esp6500siv2_all"
    ]

def get_evs_features():
    evs_val = [
        "pvs1", "ps1", "ps2", "ps3", "ps4",
        "pm1", "pm2", "pm3", "pm4", "pm5", "pm6",
        "pp1", "pp2", "pp3", "pp4",
        "ba1", "bs1", "bs2", "bs3", "bs4",
        "bp1", "bp2", "bp3", "bp4", "bp5", "bp7"
    ]

    strength_scores = {
        "Unmet": 0,
        "Supporting": 1,
        "Moderate": 2,
        "Strong": 4,
        "VeryStrong": 8,
        "StandAlone": 10
    }

    colnames = []

    for ev in evs_val:
        sign = 1 if ev.startswith("p") else -1
        scores = [0]

        for k, v in strength_scores.items():
            if k != "Unmet":
                scores.append(sign * v)

        unique_scores = sorted(list(set(scores)), reverse=(sign == 1))

        for score in unique_scores:
            colnames.append(f"{ev}_{{{score}}}")

    return colnames

Feature Processing & Dataset Preprocess

In [ ]:
def func_individual_function_dat(Clinical_dat, annovar_dat, fun_name):
    feature_list = []
    for col in fun_name:
        if col in Clinical_dat.columns:
            feature_list.append(Clinical_dat[[col]])
        else:
            feature_list.append(annovar_dat[[col]])

    return pd.concat(feature_list, axis=1).set_index(Clinical_dat['index']).astype(float)

def func_individual_evs_dat(Clinical_dat, evs_name, params):
    evs_types = {name.split('_')[0] for name in evs_name}
    rows = []

    for _, row in Clinical_dat.iterrows():
        feature_dict = {name: 0 for name in evs_name}

        for ev in evs_types:
            if ev in row and not pd.isna(row[ev]):
                value = int(row[ev])
                key = f"{ev}_{{{value}}}"
                if key in feature_dict:
                    feature_dict[key] = 1

        rows.append(feature_dict)

    evs_feature = pd.DataFrame(rows, columns=evs_name, index=Clinical_dat['index'])

    if params[0]:
        noise = np.random.normal(params[1], params[2], evs_feature.shape)
        evs_feature += noise

    return evs_feature

def preprocess_dataset(dataset, dataset_type, cutoff_smp=0.4, cutoff_feat=0.6):
    key_info_columns = ['Chr', 'Start', 'End', 'Ref', 'Alt', 'Otherinfo',
                        'Phenotype', 'platform', 'study', 'PMID', 'label',
                        'ClinVar_label', 'Explanation', 'ReVe']

    available_key_columns = [col for col in key_info_columns if col in dataset.columns]

    fun_feature = func_individual_function_dat(dataset, dataset, get_continuous_feature())
    evs_feature = func_individual_evs_dat(dataset, get_evs_features(), [False, 0, 0.02])

    feats = pd.concat([fun_feature, evs_feature], axis=1)

    for col in available_key_columns:
        if col in dataset.columns:
            feats[col] = dataset[col].values

    df = feats.copy()

    missing_per_row = df[list(fun_feature.columns) + list(evs_feature.columns)].isnull().mean(axis=1)
    df = df[missing_per_row <= cutoff_smp]

    missing_per_col = df[list(fun_feature.columns) + list(evs_feature.columns)].isnull().mean()
    removed_features = missing_per_col[missing_per_col > cutoff_feat].index.tolist()

    df = df[[c for c in df.columns if c not in removed_features]]

    df_reset = df.reset_index().rename(columns={'index': 'variant_index'})

    return df_reset

preprocessed_coding = preprocess_dataset(coding_data, "coding", 0.4, 0.7)

preprocessed_coding.to_csv(f"{outdir}/preprocessed_coding_with_variant_info.csv", index=False)

Imputation + Normalization

In [ ]:
def impute_normalize_dataset(df, dataset_type, outdir):
    labels = df['label'].copy()

    non_numeric_columns = df.select_dtypes(include=['object']).columns.tolist()
    non_numeric_data = df[non_numeric_columns].copy()

    numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
    if 'label' in numeric_columns:
        numeric_columns.remove('label')

    features = df[numeric_columns].copy()

    imputer = KNNImputer(n_neighbors=5)
    features_filled = pd.DataFrame(imputer.fit_transform(features), columns=features.columns)

    df_filled = pd.concat([features_filled, non_numeric_data, labels], axis=1)

    continuous_features = [f for f in get_continuous_feature() if f in df_filled.columns]

    if continuous_features:
        scaler = MinMaxScaler()
        df_filled[continuous_features] = scaler.fit_transform(df_filled[continuous_features])

    output_path = os.path.join(outdir, f'processed_{dataset_type}.csv')
    df_filled.to_csv(output_path, index=False)

    return df_filled

def standardize_dataset(input_path, output_path):
    df = pd.read_csv(input_path)

    non_numeric_columns = [
        'ClinVar_label', 'variant_index', 'Otherinfo', 'Phenotype',
        'platform', 'study', 'PMID', 'Explanation'
    ]
    available_non_numeric = [col for col in non_numeric_columns if col in df.columns]

    non_numeric_data = df[available_non_numeric].copy()
    labels = df['label'].copy()

    numeric_columns = [
        col for col in df.columns
        if col not in available_non_numeric and col != 'label'
    ]

    features = df[numeric_columns].copy()

    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(features)

    scaled_df = pd.DataFrame(scaled_features, columns=features.columns)

    final_df = pd.concat([scaled_df, non_numeric_data, labels], axis=1)
    final_df.to_csv(output_path, index=False)

    return final_df

final_coding = impute_normalize_dataset(preprocessed_coding, "coding", outdir)


final_scaled = standardize_dataset(final_coding, "processed_data/processed_coding_scaled.csv")